# Production Logging — Beyond Print

In [ ]:
# Day 38 — Production Logging
#
# `print("Model trained")` is fine for your laptop.
# In production, you need to know:
# - WHEN was it trained? (Timestamp)
# - WHO logged it? (Module name)
# - HOW important is this message? (Severity/Level)
#
# Without this, debugging a crashed ML pipeline in Azure or Kubernetes is a nightmare.
# Production systems ingest logs (Datadog, Splunk, ELK) to create alerts and dashboards.
#
# Today we'll cover:
#   1. The `logging` module & Log Levels
#   2. Handlers and Formatters
#   3. Rotating File Logs
#   4. JSON Structured Logging (The Gold Standard)

### 1. Basic Logging & Log Levels

In [ ]:
import logging

# Basic setup (You usually do this once at the start of your script)
# By default, Python only shows WARNING and above.
# We set it to INFO so we can see our info logs.
logging.basicConfig(level=logging.INFO)

# Create a logger for this specific module
logger = logging.getLogger("ml_trainer")

# The 5 standard levels of severity:
logger.debug("Connecting to database at 10.0.0.1...")  # Detail for devs (usually hidden)
logger.info("Started training Random Forest model.")   # Normal behavior
logger.warning("Disk space is below 20%.")             # Something might go wrong soon
logger.error("Failed to download dataset!")            # An operation failed
logger.critical("Database destroyed. Shutting down.")  # The whole app is crashing

### 2. Formatters and Handlers

In [ ]:
# A Logger creates the message.
# A Handler sends it somewhere (Console, File, Network).
# A Formatter decides what it looks like.

import logging

# 1. Create a custom logger
adv_logger = logging.getLogger("advanced_logger")
adv_logger.setLevel(logging.DEBUG)

# Clear existing handlers if running in Jupyter multiple times
if adv_logger.hasHandlers():
    adv_logger.handlers.clear()

# 2. Create Handlers
console_handler = logging.StreamHandler() # Sends to console
file_handler = logging.FileHandler("app.log") # Sends to file

# Set different levels for handlers!
# Console shows everything, file only saves WARNING and above
console_handler.setLevel(logging.DEBUG)
file_handler.setLevel(logging.WARNING)

# 3. Create a Formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# 4. Add formatter to handlers, and handlers to logger
console_handler.setFormatter(formatter)
file_handler.setFormatter(formatter)
adv_logger.addHandler(console_handler)
adv_logger.addHandler(file_handler)

adv_logger.debug("This is a debug message (console only).")
adv_logger.error("This is an error message (console AND file).")

# Check your local directory, you will see 'app.log' has been created!

### 3. Rotating File Logs

In [ ]:
# If your ML server runs for 3 months, app.log might become 100GB and crash the server.
# We use RotatingFileHandler to keep log sizes manageable.

import logging
from logging.handlers import RotatingFileHandler

rot_logger = logging.getLogger("rotator")
rot_logger.setLevel(logging.INFO)

if rot_logger.hasHandlers():
    rot_logger.handlers.clear()

# Max size 2000 bytes (~2KB), keep 3 backup files (app_rotated.log.1, .2, .3)
rot_handler = RotatingFileHandler("app_rotated.log", maxBytes=2000, backupCount=3)
rot_handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
rot_logger.addHandler(rot_handler)

# Let's flood the logger to trigger rotation
for i in range(100):
    rot_logger.info(f"Log entry number {i}: This is a long string to take up space.")

print("Check your directory! You should see app_rotated.log, app_rotated.log.1, etc.")

### 4. JSON Structured Logging

In [ ]:
# Standard text logs are hard for machines to search.
# E.g., searching for all errors from 'user_id 42' requires messy regex.
# JSON Structured Logging outputs every log line as a JSON dictionary.
# This is the standard for modern Kubernetes/Cloud deployments.
#
# We use the `python-json-logger` library.
# Install: pip install python-json-logger

import logging
from pythonjsonlogger import jsonlogger

json_logger = logging.getLogger("json_test")
json_logger.setLevel(logging.INFO)

if json_logger.hasHandlers():
    json_logger.handlers.clear()

logHandler = logging.StreamHandler()
# The format string decides which fields go into the JSON output
formatter = jsonlogger.JsonFormatter('%(asctime)s %(levelname)s %(name)s %(message)s')
logHandler.setFormatter(formatter)
json_logger.addHandler(logHandler)

# We can pass extra dictionaries directly into the JSON!
json_logger.info("Model training completed", extra={"model_name": "XGBoost", "accuracy": 0.95})
json_logger.error("API Timeout", extra={"endpoint": "/predict", "latency_ms": 5000})

# Look at the output! It's pure JSON. Systems like Datadog parse this automatically.

In [ ]:
# Practice Questions:
#
# 1. Create a logger that writes ONLY ERRORs to `errors.log`, but writes 
#    INFO and above to the console.
#
# 2. Why is RotatingFileHandler better than standard FileHandler for long-running processes?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: MLOps Standard Logger
# Create a Python module `logger_setup.py` that contains a function `get_mlops_logger(name)`.
# This function should return a logger configured to output JSON to the console
# (so Kubernetes can scrape it). It should include a custom field `environment` (e.g., "prod" or "dev").

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Observability: If you can't see what the model is doing, you can't fix it when it breaks.
# 2. JSON Logs: Centralized logging systems (Azure Monitor, ELK) index JSON logs instantly. You can query `accuracy > 0.9`.
# 3. Disk Space: Unrotated logs are a massive cause of server downtime in production.